# Notebook 02 - Tiền xử lý dữ liệu

Notebook thực hiện riêng Bước 2. Đầu vào là dataset.zip; đầu ra là preprocessing_artifacts.zip gồm schema, preprocessing pipeline và split manifest. Các nhiệm vụ được tách nhỏ để dễ theo dõi và trình bày.

## 0. Cách chạy trên Google Colab

Chọn Runtime → Run all, upload dataset.zip và tải preprocessing_artifacts.zip ở cuối. Notebook chạy độc lập, không cần giữ biến từ notebook 01.

In [1]:
from pathlib import Path
from google.colab import files
import io, json, shutil, sys, zipfile
import joblib
import numpy as np
import pandas as pd
import sklearn

ROOT=Path('/content/ckd_colab')
DATA_PATH=ROOT/'dataset.zip'
MODEL_DIR=ROOT/'models'
EXPORT_DIR=ROOT/'preprocessing_export'
for directory in [ROOT,MODEL_DIR,EXPORT_DIR]:
    directory.mkdir(parents=True,exist_ok=True)

print('Python:',sys.version.split()[0])
print('pandas:',pd.__version__,'| numpy:',np.__version__)
print('scikit-learn:',sklearn.__version__,'| joblib:',joblib.__version__)

if not DATA_PATH.is_file():
    print('Upload dataset.zip để chạy notebook 02.')
    uploaded=files.upload()
    names=[name for name in uploaded if name.lower().endswith('.zip')]
    if len(names)!=1:
        raise FileNotFoundError('Cần upload đúng một file dataset.zip.')
    DATA_PATH.write_bytes(uploaded[names[0]])
print('Dataset:',DATA_PATH)

Python: 3.13.15
pandas: 2.2.3 | numpy: 2.1.3
scikit-learn: 1.6.1 | joblib: 1.6.0
Dataset: /content/ckd_colab/dataset.zip


## 1. Khai báo bài toán và đặc trưng

Target là classification với hai lớp ckd/notckd. Cột id bị loại vì chỉ là mã định danh, không có ý nghĩa lâm sàng và có thể khiến model học thuộc từng dòng.

In [2]:
RANDOM_STATE=42
TEST_SIZE=0.20
TARGET='classification'
POSITIVE_LABEL='ckd'
NEGATIVE_LABEL='notckd'
NUMERIC_FEATURES=['age','bp','sg','al','su','bgr','bu','sc','sod','pot','hemo','pcv','wc','rc']
CATEGORICAL_FEATURES=['rbc','pc','pcc','ba','htn','dm','cad','appet','pe','ane']
FEATURES=NUMERIC_FEATURES+CATEGORICAL_FEATURES
print('Numeric:',len(NUMERIC_FEATURES),'| Categorical:',len(CATEGORICAL_FEATURES),'| Tổng:',len(FEATURES))

Numeric: 14 | Categorical: 10 | Tổng: 24


## 2. Hàm đọc và làm sạch

Quy tắc: ZIP phải có đúng một CSV; xóa tab/khoảng trắng; đưa chuỗi về chữ thường; đổi ? và chuỗi rỗng thành missing; ép đúng 14 cột numeric; xóa dòng trùng; kiểm tra target chỉ còn hai nhãn.

In [16]:
def read_zip_dataset(path):
    with zipfile.ZipFile(path) as archive:
        names=[name for name in archive.namelist() if name.lower().endswith('.csv')]
        if len(names)!=1:
            raise ValueError(f'dataset.zip phải chứa đúng 1 CSV, hiện có: {names}')
        print('Đọc CSV:',names[0])
        return pd.read_csv(io.BytesIO(archive.read(names[0])))

def clean_dataframe(dataframe):
    cleaned=dataframe.copy()
    cleaned.columns=cleaned.columns.astype(str).str.replace('\t','',regex=False).str.strip()
    missing_columns=sorted(set(FEATURES+[TARGET])-set(cleaned.columns))
    if missing_columns:
        raise ValueError(f'Thiếu cột: {missing_columns}')
    for column in cleaned.select_dtypes(include=['object','str','string']).columns:
        cleaned[column]=(cleaned[column].astype('string').str.replace('\t','',regex=False)
                         .str.strip().str.lower().replace({'?':pd.NA,'':pd.NA}))
    for column in NUMERIC_FEATURES:
        cleaned[column]=pd.to_numeric(cleaned[column],errors='coerce').astype('float64')
    for column in CATEGORICAL_FEATURES:
        cleaned[column]=cleaned[column].astype('object').where(cleaned[column].notna(),np.nan)
    cleaned[TARGET]=cleaned[TARGET].astype('string').str.strip().str.lower().astype('object')
    unexpected=sorted(set(cleaned[TARGET].dropna().unique())-{POSITIVE_LABEL,NEGATIVE_LABEL})
    if unexpected:
        raise ValueError(f'Nhãn không hợp lệ: {unexpected}')
    if cleaned[TARGET].isna().any():
        raise ValueError('Target còn missing.')
    return cleaned.drop_duplicates().reset_index(drop=True)

## 3. Kiểm tra dữ liệu thô

Kiểm tra kích thước, cột, kiểu, nhãn, trùng lặp và missing trước khi biến đổi để chứng minh vì sao cần các bước xử lý tiếp theo.

In [4]:
raw_df=read_zip_dataset(DATA_PATH)
print('Kích thước thô:',raw_df.shape)
print('Dòng trùng:',int(raw_df.duplicated().sum()))
print('Các cột:',raw_df.columns.tolist())
display(raw_df[TARGET].value_counts(dropna=False).to_frame('target_count'))
display(raw_df.isna().sum().sort_values(ascending=False).head(10).to_frame('missing_count'))
display(raw_df.dtypes.rename('dtype').to_frame())

Đọc CSV: kidney_disease.csv
Kích thước thô: (400, 26)
Dòng trùng: 0
Các cột: ['id', 'age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification']


,target_count
classification,
ckd,248
notckd,150
ckd\t,2


,missing_count
rbc,152
rc,130
wc,105
pot,88
sod,87
pcv,70
pc,65
hemo,52
su,49
sg,47


,dtype
id,int64
age,float64
bp,float64
sg,float64
al,float64
su,float64
rbc,object
pc,object
pcc,object
ba,object


## 4. Chuẩn hóa và xác nhận dữ liệu

Cell này chỉ chuẩn hóa biểu diễn/kiểu dữ liệu. Median và mode chưa được tính trên toàn bộ dữ liệu; chúng sẽ nằm trong pipeline sau khi chia train/test.

In [6]:
def clean_dataframe(df):
    df = df.copy()

    # Chuẩn hóa tên cột
    df.columns = df.columns.str.strip()

    # Chuẩn hóa text + missing
    for col in df.select_dtypes(include="object"):
        df[col] = df[col].map(
            lambda x: x.strip() if isinstance(x, str) else x
        )
        df[col] = df[col].replace(["?", "\t?", "", " "], np.nan)

    # Numeric
    for col in NUMERIC_FEATURES:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Target
    df[TARGET] = df[TARGET].map(
        lambda x: x.strip().lower() if isinstance(x, str) else x
    )

    df[TARGET] = df[TARGET].replace({
        "ckd": POSITIVE_LABEL,
        "notckd": NEGATIVE_LABEL
    })

    # Chỉ giữ target hợp lệ + xóa duplicate
    df = df[
        df[TARGET].isin([POSITIVE_LABEL, NEGATIVE_LABEL])
    ].drop_duplicates().reset_index(drop=True)

    return df

In [7]:
df=clean_dataframe(raw_df)
print('Kích thước sạch:',df.shape,'| Dòng đã loại:',len(raw_df)-len(df))
display(df[TARGET].value_counts(dropna=False).to_frame('target_count'))
missing=df[FEATURES].isna().sum().sort_values(ascending=False)
display(missing[missing>0].to_frame('missing_count'))
assert set(df[TARGET].unique())=={POSITIVE_LABEL,NEGATIVE_LABEL}
assert all(pd.api.types.is_float_dtype(df[col]) for col in NUMERIC_FEATURES)
print('Target và kiểu numeric hợp lệ: OK')

Kích thước sạch: (400, 26) | Dòng đã loại: 0


,target_count
classification,
ckd,250
notckd,150


,missing_count
rbc,152
rc,131
wc,106
pot,88
sod,87
pcv,71
pc,65
hemo,52
su,49
sg,47


Target và kiểu numeric hợp lệ: OK


## 5. Tách X/y và loại cột không dùng

X gồm đúng 24 đặc trưng. Không tự động bỏ biến vì tương quan và không xóa ngoại lai theo IQR: giá trị cực đoan y tế có thể là tín hiệu bệnh.

In [8]:
X=df[FEATURES].copy()
y=df[TARGET].copy()
assert list(X.columns)==FEATURES
assert TARGET not in X.columns and 'id' not in X.columns
print('X:',X.shape,'| y:',y.shape)
display(pd.DataFrame({'feature':FEATURES,'type':['numeric']*len(NUMERIC_FEATURES)+['categorical']*len(CATEGORICAL_FEATURES)}))

X: (400, 24) | y: (400,)


,feature,type
0,age,numeric
1,bp,numeric
2,sg,numeric
3,al,numeric
4,su,numeric
5,bgr,numeric
6,bu,numeric
7,sc,numeric
8,sod,numeric
9,pot,numeric


In [17]:
outlier_report = []

for col in NUMERIC_FEATURES:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    count = ((df[col] < lower) | (df[col] > upper)).sum()

    outlier_report.append({
        "feature": col,
        "lower": lower,
        "upper": upper,
        "outlier_count": int(count)
    })

outlier_report = pd.DataFrame(outlier_report)

display(outlier_report)

,feature,lower,upper,outlier_count
0,age,8.250,98.250,10
1,bp,55.000,95.000,36
2,sg,0.995,1.035,0
3,al,-3.000,5.000,0
4,su,0.000,0.000,61
5,bgr,3.000,259.000,34
6,bu,-31.500,124.500,38
7,sc,-1.950,5.650,51
8,sod,124.500,152.500,16
9,pot,2.150,6.550,4


Không tự động loại các giá trị ngoại lai vì trong dữ liệu y tế, giá trị cực đoan có thể phản ánh tình trạng bệnh. Do đó chỉ loại những giá trị được xác định là lỗi nhập liệu hoặc nằm ngoài miền hợp lệ của dữ liệu.

## 6. Chia train/test trước khi fit

Dùng 80/20 vì bộ dữ liệu nhỏ; stratify giữ tỷ lệ lớp; random_state=42 giúp notebook 03/04 tái tạo đúng split. Test được giữ kín khỏi mọi bước chọn tham số.

In [9]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=TEST_SIZE,random_state=RANDOM_STATE,stratify=y
)
print('Train:',X_train.shape,'| Test giữ kín:',X_test.shape)
display(pd.DataFrame({
    'train_count':y_train.value_counts(),
    'train_percent':y_train.value_counts(normalize=True).mul(100).round(2),
    'test_count':y_test.value_counts(),
    'test_percent':y_test.value_counts(normalize=True).mul(100).round(2)
}))

Train: (320, 24) | Test giữ kín: (80, 24)


,train_count,train_percent,test_count,test_percent
classification,,,,
ckd,200,62.5,50,62.5
notckd,120,37.5,30,37.5


## 7. Pipeline numeric

Median bền vững hơn mean khi có ngoại lai. StandardScaler cần cho Logistic Regression, KNN và SVM. Các thống kê chỉ được học từ train.

In [10]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
numeric_pipeline=Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])
print(numeric_pipeline)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])


## 8. Pipeline categorical

Mode điền category phổ biến; One-Hot phù hợp vì category không có thứ tự; handle_unknown=ignore tránh lỗi khi inference gặp category mới.

In [11]:
from sklearn.preprocessing import OneHotEncoder
categorical_pipeline=Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder',OneHotEncoder(handle_unknown='ignore'))
])
print(categorical_pipeline)

Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore'))])


## 9. Ghép bằng ColumnTransformer

Mỗi nhóm cột đi qua đúng pipeline. Tất cả model ở bước 3 sẽ dùng cùng cấu trúc này để so sánh công bằng.

In [12]:
from sklearn.compose import ColumnTransformer
preprocessor=ColumnTransformer([
    ('numeric',numeric_pipeline,NUMERIC_FEATURES),
    ('categorical',categorical_pipeline,CATEGORICAL_FEATURES)
])
print(preprocessor)

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu',
                                  'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc',
                                  'rc']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad',
                                  'appet', 'pe', 'ane'])])


## 10. Fit chỉ trên train và kiểm tra leakage

Preprocessor được fit trên X_train. X_test chỉ transform để xác nhận số cột và missing, không tham gia học median/mode/scaler/encoder.

In [13]:
preprocessor.fit(X_train,y_train)
train_transformed=preprocessor.transform(X_train)
test_transformed=preprocessor.transform(X_test)
train_values=train_transformed.data if hasattr(train_transformed,'data') else np.asarray(train_transformed)
test_values=test_transformed.data if hasattr(test_transformed,'data') else np.asarray(test_transformed)
assert list(preprocessor.feature_names_in_)==FEATURES
assert train_transformed.shape[1]==test_transformed.shape[1]
assert np.isfinite(train_values).all() and np.isfinite(test_values).all()
print('Train transform:',train_transformed.shape,'| Test transform:',test_transformed.shape)
print('Pipeline chỉ fit trên train, không còn NaN/Inf: OK')

Train transform: (320, 34) | Test transform: (80, 34)
Pipeline chỉ fit trên train, không còn NaN/Inf: OK


## 11. Tạo schema.json

Schema là hợp đồng cho Frontend, Backend và AI Service: tên/thứ tự/kiểu feature, range, category, nhãn và cấu hình split. Range/category chỉ dùng validation, không phải thống kê model học.

In [14]:
schema={
 'task_type':'binary_classification','target':TARGET,'positive_label':POSITIVE_LABEL,
 'target_labels':[POSITIVE_LABEL,NEGATIVE_LABEL],'features':FEATURES,
 'numeric_features':NUMERIC_FEATURES,'categorical_features':CATEGORICAL_FEATURES,
 'allowed_categories':{col:sorted(df[col].dropna().astype(str).unique().tolist()) for col in CATEGORICAL_FEATURES},
 'numeric_ranges':{col:{'min':float(df[col].min()),'max':float(df[col].max())} for col in NUMERIC_FEATURES},
 'test_size':TEST_SIZE,'random_state':RANDOM_STATE
}
schema_path=MODEL_DIR/'schema.json'
schema_path.write_text(json.dumps(schema,ensure_ascii=False,indent=2),encoding='utf-8')
assert schema['features']==list(preprocessor.feature_names_in_)
print('Đã lưu:',schema_path)
print(json.dumps(schema,ensure_ascii=False,indent=2))

Đã lưu: /content/ckd_colab/models/schema.json
{
  "task_type": "binary_classification",
  "target": "classification",
  "positive_label": "ckd",
  "target_labels": [
    "ckd",
    "notckd"
  ],
  "features": [
    "age",
    "bp",
    "sg",
    "al",
    "su",
    "bgr",
    "bu",
    "sc",
    "sod",
    "pot",
    "hemo",
    "pcv",
    "wc",
    "rc",
    "rbc",
    "pc",
    "pcc",
    "ba",
    "htn",
    "dm",
    "cad",
    "appet",
    "pe",
    "ane"
  ],
  "numeric_features": [
    "age",
    "bp",
    "sg",
    "al",
    "su",
    "bgr",
    "bu",
    "sc",
    "sod",
    "pot",
    "hemo",
    "pcv",
    "wc",
    "rc"
  ],
  "categorical_features": [
    "rbc",
    "pc",
    "pcc",
    "ba",
    "htn",
    "dm",
    "cad",
    "appet",
    "pe",
    "ane"
  ],
  "allowed_categories": {
    "rbc": [
      "abnormal",
      "normal"
    ],
    "pc": [
      "abnormal",
      "normal"
    ],
    "pcc": [
      "notpresent",
      "present"
    ],
    "ba": [
      "notpresen

## 12. Lưu artefact bước 2

preprocessor.joblib chứng minh pipeline serialize được. split_manifest ghi quy tắc chia. Ở GridSearchCV, bước 3 vẫn tạo preprocessor mới trong từng fold để tránh leakage giữa fold.

In [15]:
preprocessor_path=MODEL_DIR/'preprocessor.joblib'
joblib.dump(preprocessor,preprocessor_path,compress=3)
manifest_path=MODEL_DIR/'split_manifest.json'
manifest_path.write_text(json.dumps({
 'train_rows':len(X_train),'test_rows':len(X_test),'test_size':TEST_SIZE,
 'stratified':True,'random_state':RANDOM_STATE,'feature_count':len(FEATURES),
 'leakage_check':'preprocessor fitted only on X_train'
},ensure_ascii=False,indent=2),encoding='utf-8')
for artifact in [schema_path,preprocessor_path,manifest_path]:
    shutil.copy2(artifact,EXPORT_DIR/artifact.name)
archive=shutil.make_archive(str(ROOT/'preprocessing_artifacts'),'zip',EXPORT_DIR)
print('Đã tạo:',archive)
files.download(archive)

Đã tạo: /content/ckd_colab/preprocessing_artifacts.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 13. Kết luận

Dữ liệu đã đúng nhãn/kiểu; id bị loại; chia trước khi fit; missing/encoding/scaling nằm trong pipeline; schema khớp 24 feature; test chưa được dùng để chọn model.